# NBA Model Training on Colab GPU

**Seamless workflow:**
1. Clone code from GitHub
2. Mount Google Drive for persistence
3. Configure & train
4. Download models/data

---
**Auto-Sized Models:**
The training script auto-detects your GPU (H100, A100, T4, etc.) and sizes models optimally.
Bigger GPU = Bigger models = Better predictions.

---
**GPU Optimizations v2.0:**
- TF32 acceleration on Ampere+ GPUs (RTX 30xx, A100, H100)
- BF16 mixed precision on supported hardware
- torch.compile for PyTorch 2.0+ speedup
- Optimal DataLoader worker configuration
- Gradient accumulation for large batch training

---
**Drive structure after training:**
```
/MyDrive/nba_model/
├── data/              # Cached NBA data
├── models/            # Trained models
└── training_config.json  # Model config used
```

In [ ]:
# @title Cell 1: Setup Environment
# @markdown Run this cell first to set up everything

import os
import sys
import subprocess

# 1. Clone repository (change URL to your fork if needed)
REPO_URL = "https://github.com/jxylxnn/knowing.git"  # @param {type:"string"}
REPO_NAME = "knowing"

if not os.path.exists(REPO_NAME):
    print(f"Cloning {REPO_URL}...")
    !git clone {REPO_URL}
else:
    print(f"Repository already exists, pulling latest...")
    !cd {REPO_NAME} && git pull

os.chdir(REPO_NAME)
print(f"Working directory: {os.getcwd()}")

# 2. Mount Google Drive
from google.colab import drive
print("\nMounting Google Drive...")
drive.mount('/content/drive')

# 3. Create persistent directories in Drive
DRIVE_BASE = "/content/drive/MyDrive/nba_model"
DRIVE_DATA = f"{DRIVE_BASE}/data"
DRIVE_MODELS = f"{DRIVE_BASE}/models"

os.makedirs(DRIVE_DATA, exist_ok=True)
os.makedirs(DRIVE_MODELS, exist_ok=True)
print(f"Drive directories ready:\n  Data: {DRIVE_DATA}\n  Models: {DRIVE_MODELS}")

# 4. Detect GPU and install correct PyTorch version
print("\n" + "="*60)
print("GPU DETECTION & PYTORCH SETUP")
print("="*60)

import torch

# Check if CUDA is already available
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    compute_cap = torch.cuda.get_device_capability(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✓ CUDA PyTorch already installed!")
    print(f"  GPU: {gpu_name}")
    print(f"  Compute Capability: {compute_cap[0]}.{compute_cap[1]}")
    print(f"  VRAM: {vram:.1f} GB")
else:
    print("CUDA not available with current PyTorch. Reinstalling with CUDA...")
    
    # Detect GPU architecture from nvidia-smi
    try:
        result = subprocess.run(
            ['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'],
            capture_output=True, text=True
        )
        if result.returncode == 0:
            compute_cap_raw = result.stdout.strip().split('\n')[0]
            compute_cap_val = int(compute_cap_raw.replace('.', ''))
        else:
            compute_cap_val = 75  # Default for T4
    except:
        compute_cap_val = 75  # Default for T4
    
    # Select PyTorch index based on GPU architecture
    if compute_cap_val >= 120:  # Blackwell (RTX 50-series)
        pytorch_index = "https://download.pytorch.org/whl/nightly/cu124"
        print("  Detected Blackwell GPU - using PyTorch nightly")
    elif compute_cap_val >= 80:  # Ampere/Ada (A100, RTX 30/40)
        pytorch_index = "https://download.pytorch.org/whl/cu121"
        print("  Detected Ampere/Ada GPU - using CUDA 12.1")
    else:  # Turing/Volta (T4, V100)
        pytorch_index = "https://download.pytorch.org/whl/cu118"
        print("  Detected Turing/Volta GPU - using CUDA 11.8")
    
    # Uninstall CPU-only torch
    print("\nUninstalling CPU-only PyTorch...")
    !pip uninstall -y torch torchvision torchaudio > /dev/null 2>&1
    
    # Install CUDA-enabled PyTorch
    print(f"Installing CUDA PyTorch from {pytorch_index}...")
    !pip install -q torch torchvision torchaudio --index-url {pytorch_index}
    
    # Reimport and verify
    import importlib
    importlib.reload(torch)
    
    if torch.cuda.is_available():
        print(f"\n✓ CUDA PyTorch installed successfully!")
        print(f"  GPU: {torch.cuda.get_device_name(0)}")
        print(f"  CUDA Version: {torch.version.cuda}")
    else:
        print("\n⚠ CUDA still not available. Check Colab runtime type (Runtime → Change runtime type → GPU)")

# 5. Install other dependencies
print("\nInstalling other dependencies...")
!pip install -q catboost lightgbm xgboost scikit-learn pandas numpy tqdm nba_api joblib pyyaml psutil rich

# 6. Show GPU info and optimization capabilities
print("\n" + "="*60)
print("GPU OPTIMIZATION CAPABILITIES")
print("="*60)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    compute_cap = torch.cuda.get_device_capability(0)
    
    print(f"GPU: {gpu_name}")
    print(f"VRAM: {vram:.1f} GB")
    print(f"Compute Capability: {compute_cap[0]}.{compute_cap[1]}")
    print(f"PyTorch: {torch.__version__}")
    print(f"CUDA: {torch.version.cuda}")
    print()
    
    # Check optimization capabilities
    ampere_plus = compute_cap[0] >= 8
    
    print("Optimization Support:")
    print(f"  TF32 (2-8x speedup): {'✓ ENABLED' if ampere_plus else '✗ Not available'}")
    print(f"  BF16 Mixed Precision: {'✓ ENABLED' if ampere_plus else '✗ Not available'}")
    print(f"  FP16 Mixed Precision: ✓ ENABLED")
    print(f"  torch.compile: {'✓ ENABLED' if hasattr(torch, 'compile') else '✗ PyTorch < 2.0'}")
    
    # Estimate model size
    if 'H100' in gpu_name or vram > 70:
        model_size = "ULTRA (H100-class)"
    elif 'A100' in gpu_name:
        model_size = "LARGE (A100)" if vram < 50 else "ULTRA (A100-80GB)"
    elif 'L4' in gpu_name or vram > 20:
        model_size = "MEDIUM+ (L4)"
    elif 'T4' in gpu_name or 'V100' in gpu_name:
        model_size = "MEDIUM (T4/V100)"
    else:
        model_size = "SMALL (Unknown GPU)"
    print(f"\nExpected Model Size: {model_size}")
    print(f"\n✓ GPU acceleration ENABLED with optimizations")
else:
    print("⚠ No GPU detected - will use CPU (training will be SLOW)")
    print("  Go to: Runtime → Change runtime type → GPU")
print("="*60)

In [ ]:
# @title Cell 2: Configure Training
# @markdown ---
# @markdown **Training Mode:**
training_mode = "fetch_and_train"  # @param ["fetch_and_train", "use_cached_train", "train_only"]
# @markdown * `fetch_and_train` - Fetch fresh data, then train (first run)
# @markdown * `use_cached_train` - Use cached data from Drive, then train
# @markdown * `train_only` - Skip data fetch, just train (if data already in Drive)
# @markdown ---
# @markdown **Model Size Override (leave as 'auto' for auto-detection):**
model_size_override = "auto"  # @param ["auto", "small", "medium", "large", "pro", "ultra"]
# @markdown * `auto` - Detect GPU and size automatically (recommended)
# @markdown * `small` - Force small model (CPU or weak GPU)
# @markdown * `medium` - Force medium model (T4, V100)
# @markdown * `large` - Force large model (A100-40GB)
# @markdown * `pro` - Force pro model (A100-80GB)
# @markdown * `ultra` - Force ultra model (H100)
# @markdown ---
# @markdown **Data Fetching** (ignored if using cached data):
seasons_mode = "update"  # @param ["update", "recent", "current", "all", "custom"]
custom_seasons_input = ""  # @param {type:"string"}
force_refresh = False  # @param {type:"boolean"}
# @markdown * `update` - Fetch only new games since last run (fast, ~1 min)
# @markdown * `recent` - Last 10 seasons (recommended for first run)
# @markdown * `current` - Current season only (fast, ~5 min)
# @markdown * `all` - All NBA seasons since 1946 (slow, ~2 hours)
# @markdown * `custom` - Specify below (e.g., "2023-24,2024-25" or "70-80")

# Store config for use in next cells
config = {
    'training_mode': training_mode,
    'model_size': model_size_override,
    'seasons_mode': seasons_mode,
    'custom_seasons': custom_seasons_input,
    'force_refresh': force_refresh,
    'drive_data': DRIVE_DATA,
    'drive_models': DRIVE_MODELS
}
print("Configuration saved. Proceed to Cell 3.")

In [ ]:
# @title Cell 3: Fetch Data (if needed)

import subprocess
import shutil

# Check if we need to fetch data
need_fetch = config['training_mode'] == 'fetch_and_train'
use_cached = config['training_mode'] in ['use_cached_train', 'train_only']

# Check for cached data
cached_data_exists = os.path.exists(config['drive_data']) and len(os.listdir(config['drive_data'])) > 0

if use_cached and not cached_data_exists:
    print("WARNING: No cached data found in Drive. Switching to fetch_and_train mode.")
    need_fetch = True
    use_cached = False

if need_fetch:
    cmd = [sys.executable, "update_data.py", "--data-dir", config['drive_data']]
    
    mode = config['seasons_mode']
    
    if mode == "update":
        cmd.append("--update")
    elif mode == "recent":
        cmd.append("--all-seasons")
    elif mode == "all":
        cmd.append("--full-scrape")
    elif mode == "current":
        cmd.append("--current-season")
    elif mode == "custom" and config['custom_seasons'].strip():
        seasons = []
        for part in config['custom_seasons'].split(','):
            part = part.strip()
            if '-' in part:
                start, end = part.split('-', 1)
                try:
                    start_idx = int(start.strip())
                    end_idx = int(end.strip())
                    for i in range(start_idx, end_idx + 1):
                        seasons.append(str(i))
                except:
                    pass
            else:
                seasons.append(part)
        
        ALL_SEASONS = [
            '1946-47', '1947-48', '1948-49', '1949-50',
            '1950-51', '1951-52', '1952-53', '1953-54',
            '1954-55', '1955-56', '1956-57', '1957-58',
            '1958-59', '1959-60', '1960-61', '1961-62',
            '1962-63', '1963-64', '1964-65', '1965-66',
            '1966-67', '1967-68', '1968-69', '1969-70',
            '1970-71', '1971-72', '1972-73', '1973-74',
            '1974-75', '1975-76', '1976-77', '1977-78',
            '1978-79', '1979-80', '1980-81', '1981-82',
            '1982-83', '1983-84', '1984-85', '1985-86',
            '1986-87', '1987-88', '1988-89', '1989-90',
            '1990-91', '1991-92', '1992-93', '1993-94',
            '1994-95', '1995-96', '1996-97', '1997-98',
            '1998-99', '1999-00', '2000-01', '2001-02',
            '2002-03', '2003-04', '2004-05', '2005-06',
            '2006-07', '2007-08', '2008-09', '2009-10',
            '2010-11', '2011-12', '2012-13', '2013-14',
            '2014-15', '2015-16', '2016-17', '2017-18',
            '2018-19', '2019-20', '2020-21', '2021-22',
            '2022-23', '2023-24', '2024-25', '2025-26'
        ]
        
        final_seasons = []
        for s in seasons:
            try:
                idx = int(s)
                if 1 <= idx <= len(ALL_SEASONS):
                    final_seasons.append(ALL_SEASONS[idx - 1])
            except:
                if s in ALL_SEASONS:
                    final_seasons.append(s)
        
        for s in final_seasons:
            cmd.extend(["--season", s])
    
    if config['force_refresh']:
        cmd.append("--force")
    
    print(f"Running: {' '.join(cmd)}")
    print("\n" + "="*50)
    print("Fetching NBA data...")
    print("="*50 + "\n")
    
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='')
    process.wait()
    print("\nData fetch complete!")

elif use_cached:
    print(f"Using cached data from: {config['drive_data']}")
    print(f"Files: {os.listdir(config['drive_data'])}")

else:
    print("Skipping data fetch (train_only mode with existing data)")

In [ ]:
# @title Cell 4: Train Models (with GPU Optimizations)
# @markdown Train models using the new efficient pipeline v2.0 with GPU optimizations.
# @markdown ---
# @markdown **Training Mode:**
training_pipeline_mode = "standard"  # @param ["quick", "standard", "full"]
# @markdown * `quick` - Fast training for testing (500 iterations, ~5 min)
# @markdown * `standard` - Full training with all models (3000 iterations, ~15 min)
# @markdown * `full` - Extended training for max accuracy (5000 iterations, ~30 min)
# @markdown ---
# @markdown **Enable Parallel Training:**
enable_parallel = True  # @param {type:"boolean"}
# @markdown Parallel training trains all 6 targets simultaneously (3x faster!)
# @markdown ---
# @markdown **GPU Optimizations (auto-enabled on supported hardware):**
# @markdown - TF32: 2-8x speedup on Ampere+ GPUs (RTX 30xx, A100, H100)
# @markdown - BF16: 50% memory reduction + 15% speedup on Ampere+
# @markdown - torch.compile: 10-40% speedup on PyTorch 2.0+
# @markdown - 8 DataLoader workers for optimal throughput

model_size_arg = config['model_size']
print("\n" + "="*60)
print("NBA MODEL TRAINER v2.0 - GPU OPTIMIZED PIPELINE")
print("="*60)
print(f"Training Mode: {training_pipeline_mode.upper()}")
print(f"Parallel Training: {'ENABLED' if enable_parallel else 'DISABLED'}")
print(f"Data directory: {config['drive_data']}")
print(f"Models directory: {config['drive_models']}")
if model_size_arg != 'auto':
    print(f"Model size override: {model_size_arg}")
else:
    print("Model size: AUTO (will detect GPU capabilities)")

# Print optimization info
import torch
if torch.cuda.is_available():
    compute_cap = torch.cuda.get_device_capability(0)
    print(f"\nGPU Optimizations Available:")
    print(f"  TF32: {'ENABLED' if compute_cap[0] >= 8 else 'N/A (requires Ampere+)'}")
    print(f"  BF16: {'ENABLED' if compute_cap[0] >= 8 else 'N/A (requires Ampere+)'}")
    print(f"  torch.compile: {'ENABLED' if hasattr(torch, 'compile') else 'N/A (PyTorch < 2.0)'}")
print("="*60 + "\n")

# Build command with new training options
train_cmd = f"python train.py --data-dir \"{config['drive_data']}\" --models-dir \"{config['drive_models']}\" --mode {training_pipeline_mode} --model-size {model_size_arg}"

if enable_parallel:
    train_cmd += " --parallel"

print(f"Executing: {train_cmd}\n")
!{train_cmd}

print("\n" + "="*60)
print("TRAINING COMPLETE!")
print("="*60)
print("\nModels saved to Google Drive:")
print(f"  {config['drive_models']}")
print("\nExperiment tracking available in:")
print(f"  /content/drive/MyDrive/nba_model/experiments/")

In [ ]:
# @title Cell 5: Download Results
# @markdown Select what to download to your local machine:

download_models = True   # @param {type:"boolean"}
download_data = False   # @param {type:"boolean"}
download_config = True  # @param {type:"boolean"}

from google.colab import files as dl_files

downloads = []

if download_models:
    models_path = config['drive_models']
    if os.path.exists(models_path) and os.listdir(models_path):
        print("Zipping models...")
        !zip -rq models.zip "{models_path}"
        dl_files.download('models.zip')
        downloads.append('models.zip')
    else:
        print("WARNING: No models found to download")

if download_data:
    data_path = config['drive_data']
    if os.path.exists(data_path) and os.listdir(data_path):
        print("Zipping data...")
        !zip -rq data.zip "{data_path}"
        dl_files.download('data.zip')
        downloads.append('data.zip')
    else:
        print("WARNING: No data found to download")

if download_config:
    config_path = os.path.join(config['drive_models'], 'training_config.json')
    if os.path.exists(config_path):
        dl_files.download(config_path)
        downloads.append('training_config.json')

if downloads:
    print(f"\nDownloaded: {', '.join(downloads)}")
else:
    print("\nNothing selected for download.")

In [ ]:
# @title Cell 5.5: Generate Player Projections
# @markdown Run simulation to generate player projections for querying.

sim_mode = "today"  # @param ["today", "date", "week"]
sim_date = ""       # @param {type:"string"}
num_sims = 100     # @param {type:"integer"}

if 'config' in globals() and config:
    _data_dir = config['drive_data']
    _models_dir = config['drive_models']
else:
    _data_dir = '/content/drive/MyDrive/nba_model/data'
    _models_dir = '/content/drive/MyDrive/nba_model/models'

cmd = f"python simulate_season.py --data-dir \"{_data_dir}\" --models-dir \"{_models_dir}\" --sims {num_sims}"

if sim_mode == 'today':
    cmd += " --today"
elif sim_mode == 'date' and sim_date:
    cmd += f" --date {sim_date}"
elif sim_mode == 'week':
    cmd += " --week"

print(f"Running: {cmd}\n")
print("="*60)
print("GENERATING PLAYER PROJECTIONS")
print("="*60)
!{cmd}
print("\n" + "="*60)
print("PROJECTIONS COMPLETE")
print("="*60)

---
# Query Player Projections

Use trained models to query player stat projections and over/under probabilities.

**Prerequisites:**
1. Run through training cells (1-4) first
2. Run Cell 5.5 to generate projections

In [ ]:
# @title Cell 6: Setup Query Environment
# @markdown Initialize the query system with trained models and data.

import os
import sys
sys.path.insert(0, os.getcwd())

from src.query.projection_loader import ProjectionLoader
from src.query.probability_calculator import ProbabilityCalculator, ProbabilityResult
from src.query.interactive_cli import InteractiveCLI

if 'config' in globals() and config:
    _drive_data = config['drive_data']
else:
    _drive_data = '/content/drive/MyDrive/nba_model/data'

QUERY_DATA_DIR = _drive_data + '/sim_results'
os.makedirs(QUERY_DATA_DIR, exist_ok=True)

query_loader = ProjectionLoader(data_dir=QUERY_DATA_DIR)
query_calculator = ProbabilityCalculator()

projections_df = query_loader.load_projections()
if projections_df.empty:
    print("No cached projections found.")
    print("  Run: !python simulate_season.py --today --data-dir \"" + _drive_data + "\"")
    print("  Or generate projections after training.")
else:
    cache_info = query_loader.get_cache_info()
    print(f"Loaded {cache_info['num_projections']} projections")
    print(f"  Data dir: {QUERY_DATA_DIR}")

In [ ]:
# @title Cell 7: Query Helper Functions
# @markdown Run this cell to define query functions.

def _ensure_loader():
    """Ensure query_loader and query_calculator are initialized."""
    global query_loader, query_calculator, QUERY_DATA_DIR
    if 'query_loader' not in globals() or query_loader is None:
        from src.query.projection_loader import ProjectionLoader
        from src.query.probability_calculator import ProbabilityCalculator
        if 'QUERY_DATA_DIR' not in globals():
            if 'config' in globals() and config:
                QUERY_DATA_DIR = config['drive_data'] + '/sim_results'
            else:
                QUERY_DATA_DIR = '/content/drive/MyDrive/nba_model/data/sim_results'
        query_loader = ProjectionLoader(data_dir=QUERY_DATA_DIR)
        query_calculator = ProbabilityCalculator()
    return query_loader

def query_player(player_name, stat='pts', line=None, opponent=None, num_sims=100):
    """
    Query over/under probability for a player stat.
    
    Args:
        player_name: Player name (e.g., "LeBron James", "Jokic", "Curry")
        stat: Stat type - 'pts', 'reb', 'ast', 'stl', 'blk', 'tov'
        line: Over/under line (auto-detected if None)
        opponent: Opponent team abbreviation (optional)
        num_sims: Number of Monte Carlo simulations
    
    Returns:
        ProbabilityResult with prob_over, prob_under, recommendation, etc.
    """
    _ensure_loader()
    projection = query_loader.find_player(player_name=player_name, opponent=opponent)
    
    if projection is None:
        print(f"Player '{player_name}' not found in projections.")
        print("Use list_players() to see available players.")
        return None
    
    if line is None:
        mean_val = projection.get_stat_mean(stat)
        line = round(mean_val - 0.5) + 0.5
    
    result = query_calculator.calculate_from_projection(
        player_name=projection.player_name,
        stat=stat,
        line=line,
        mean=projection.get_stat_mean(stat),
        std=projection.get_stat_std(stat),
        ci_low=projection.get_stat_ci(stat)[0],
        ci_high=projection.get_stat_ci(stat)[1],
        opponent=projection.opponent,
        date=projection.date,
        play_probability=projection.play_probability,
        num_sims=num_sims
    )
    result.team = projection.team
    result.is_home = projection.is_home
    
    return result


def get_projection(player_name, stat=None, opponent=None):
    """
    Get full projection for a player.
    
    Args:
        player_name: Player name
        stat: Specific stat or None for all (pts, reb, ast)
        opponent: Opponent team abbreviation (optional)
    
    Returns:
        PlayerProjection object with mean, std, ci for each stat
    """
    _ensure_loader()
    projection = query_loader.find_player(player_name=player_name, opponent=opponent)
    
    if projection is None:
        print(f"Player '{player_name}' not found.")
        return None
    
    print(f"\n{'-'*50}")
    print(f"{projection.player_name} ({projection.team} vs {projection.opponent})")
    print(f"Date: {projection.date}")
    if projection.play_probability < 1.0:
        print(f"Play Probability: {projection.play_probability*100:.0f}%")
    print(f"{'-'*50}")
    
    if stat:
        mean = projection.get_stat_mean(stat)
        std = projection.get_stat_std(stat)
        ci = projection.get_stat_ci(stat)
        print(f"\n{stat.upper()}: {mean:.1f} +/- {std:.2f}")
        print(f"95% CI: {ci[0]:.1f} - {ci[1]:.1f}")
    else:
        print(f"\n{'Stat':<12} {'Mean':>8} {'Mode':>8} {'95% CI':>18}")
        print(f"{'-'*50}")
        for s in ['pts', 'reb', 'ast']:
            mean = projection.get_stat_mean(s)
            mode = getattr(projection, f'{s}_mode', mean)
            ci = projection.get_stat_ci(s)
            print(f"{s.upper():<12} {mean:>8.1f} {mode:>8.1f} ({ci[0]:.1f} - {ci[1]:.1f})")
    print(f"{'-'*50}\n")
    
    return projection


def compare_players(players, stat='pts', opponent=None):
    """
    Compare projections for multiple players.
    
    Args:
        players: List of player names
        stat: Stat to compare
        opponent: Opponent team (optional)
    """
    if isinstance(players, str):
        print("Pass a list of players: compare_players(['LeBron', 'Curry'], 'pts')")
        return
    
    _ensure_loader()
    print(f"\n{'='*60}")
    print(f"Comparing {len(players)} players - {stat.upper()}")
    print(f"{'='*60}")
    
    results = []
    for p in players:
        proj = query_loader.find_player(p, opponent=opponent)
        if proj:
            results.append({
                'name': proj.player_name,
                'mean': proj.get_stat_mean(stat),
                'std': proj.get_stat_std(stat),
                'ci': proj.get_stat_ci(stat)
            })
        else:
            print(f"Warning: {p} not found")
    
    if not results:
        return
    
    print(f"\n{'Player':<25} {'Mean':>8} {'Std':>8} {'95% CI':>20}")
    print(f"{'-'*60}")
    for r in results:
        ci_str = f"({r['ci'][0]:.1f} - {r['ci'][1]:.1f})"
        print(f"{r['name']:<25} {r['mean']:>8.1f} {r['std']:>8.2f} {ci_str:>20}")
    print()


def list_players():
    """List all available players in projections."""
    _ensure_loader()
    players = query_loader.get_available_players()
    if not players:
        print("No players available. Generate projections first.")
        return
    print(f"\n{'='*50}")
    print(f"Available Players ({len(players)})")
    print(f"{'='*50}")
    for i, p in enumerate(players):
        if i % 3 == 0:
            print()
        print(f"  {p:<20}", end='')
        if (i + 1) % 3 == 0:
            print()
    print("\n")


def list_teams():
    """List all available teams in projections."""
    _ensure_loader()
    teams = query_loader.get_available_teams()
    if not teams:
        print("No teams available.")
        return
    print(f"\n{'='*50}")
    print(f"Available Teams ({len(teams)})")
    print(f"{'='*50}")
    print("  " + "  ".join(teams))
    print()


def print_result(result):
    """Pretty print a query result."""
    if result is None:
        return
    print(f"\n{'='*50}")
    print(f"{result.player_name} - {result.stat.upper()}")
    print(f"{'='*50}")
    print(f"  Line: {result.line}")
    print(f"  Mean: {result.mean:.1f} +/- {result.std:.2f}")
    print(f"  95% CI: {result.ci_low:.1f} - {result.ci_high:.1f}")
    print(f"{'-'*50}")
    print(f"  OVER:  {result.prob_over*100:.1f}%")
    print(f"  UNDER: {result.prob_under*100:.1f}%")
    print(f"{'-'*50}")
    print(f"  Recommendation: {result.recommendation}")
    if result.team:
        print(f"  Team: {result.team} vs {result.opponent}")
    print(f"{'='*50}\n")

print("Query functions loaded.")
print("\nAvailable functions:")
print("  query_player(name, stat='pts', line=25.5, opponent=None)")
print("  get_projection(name, stat=None, opponent=None)")
print("  compare_players(['Player1', 'Player2'], stat='pts')")
print("  list_players()")
print("  list_teams()")
print("  print_result(result)")

In [ ]:
# @title Cell 8: Example Queries
# @markdown Run examples or modify for your own queries.

example_player = "LeBron James"  # @param {type:"string"}
example_stat = "pts"            # @param ["pts", "reb", "ast", "stl", "blk", "tov"]
example_line = 25.5             # @param {type:"number"}
example_opponent = ""           # @param {type:"string"}

print("="*60)
print("QUERY EXAMPLE")
print("="*60)

if example_opponent.strip():
    result = query_player(example_player, stat=example_stat, line=example_line, opponent=example_opponent)
else:
    result = query_player(example_player, stat=example_stat, line=example_line)

if result:
    print_result(result)
else:
    print(f"\n'{example_player}' not found. Listing available players:\n")
    list_players()

In [ ]:
# @title Cell 9: Get Full Projection
# @markdown View all stats for a player.

projection_player = "Jokic"  # @param {type:"string"}

proj = get_projection(projection_player)

In [ ]:
# @title Cell 10: Compare Players
# @markdown Compare multiple players side by side.

compare_list = "LeBron James, Jokic, Curry"  # @param {type:"string"}
compare_stat = "pts"                      # @param ["pts", "reb", "ast", "stl", "blk", "tov"]

players = [p.strip() for p in compare_list.split(',')]
compare_players(players, stat=compare_stat)

In [ ]:
# @title Cell 11: Shell Command Query (Alternative)
# @markdown Run query_prob.py directly as a shell command.

shell_player = "Curry"  # @param {type:"string"}
shell_stat = "pts"     # @param ["pts", "reb", "ast", "stl", "blk", "tov"]
shell_line = 25.5      # @param {type:"number"}
shell_json = True      # @param {type:"boolean"}

# Resolve QUERY_DATA_DIR if not already defined
if 'QUERY_DATA_DIR' not in globals():
    if 'config' in globals() and config:
        QUERY_DATA_DIR = config['drive_data'] + '/sim_results'
    else:
        QUERY_DATA_DIR = '/content/drive/MyDrive/nba_model/data/sim_results'

cmd = f"python query_prob.py -p \"{shell_player}\" -s {shell_stat} -l {shell_line} --data-dir \"{QUERY_DATA_DIR}\""
if shell_json:
    cmd += " --json"

print(f"Running: {cmd}\n")
!{cmd}